# LLM Comparison: multi-model extraction benchmark

Side-by-side extraction quality and speed comparison using the 4-agent pipeline.

| Model (config name) | HuggingFace repo (Apple Silicon) | HuggingFace repo (other) |
|---|---|---|
| `mistral:7b` | `mlx-community/Mistral-7B-Instruct-v0.3-4bit` | `mistralai/Mistral-7B-Instruct-v0.3` |
| `qwen2.5:7b-instruct-q4_K_M` | `mlx-community/Qwen2.5-7B-Instruct-4bit` | `Qwen/Qwen2.5-7B-Instruct` |

Models are downloaded automatically from HuggingFace on first use and cached in `~/.cache/huggingface/hub`.
No Ollama required.


## Setup

In [1]:
import sys
from pathlib import Path

# Make sure the local src is on the path when running from the notebooks/ directory
repo_root = Path().resolve().parent  # finamt/notebooks -> finamt/
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import platform
import time
import warnings
import pandas as pd

warnings.filterwarnings("ignore")

from finamt import FinanceAgent
from finamt.agents.config import AgentsConfig
from finamt.agents import llm_backend
from huggingface_hub import scan_cache_dir, snapshot_download

IS_APPLE_SILICON = sys.platform == "darwin" and platform.machine() == "arm64"
print(f"finamt imported successfully")
print(f"Platform : {sys.platform} / {platform.machine()}")
print(f"Backend  : {'mlx-lm (Apple Silicon)' if IS_APPLE_SILICON else 'transformers'}")


finamt imported successfully
Platform : darwin / arm64
Backend  : mlx-lm (Apple Silicon)


In [2]:
# ── Receipt files ────────────────────────────────────────────────────────────
DATA_DIR = repo_root / "data"
receipts = sorted(DATA_DIR.glob("*.pdf"))
print(f"Found {len(receipts)} PDF(s) in {DATA_DIR}:")
for r in receipts:
    print(f"  {r.name}")

# ── Models to compare ────────────────────────────────────────────────────────
# Keys are the short config names recognised by llm_backend._REGISTRY.
# Remove any entry you don't want to test.
MODELS = {
    "mistral:7b":                  AgentsConfig(agent_model="mistral:7b"),
    "qwen2.5:7b-instruct-q4_K_M": AgentsConfig(agent_model="qwen2.5:7b-instruct-q4_K_M"),
}


Found 5 PDF(s) in /Users/yauheniya/Code/finamt/finamt/data:
  adobe-2601.pdf
  anthropic-2601.pdf
  github-2601.pdf
  microsoft-2601.pdf
  openai-2601.pdf


## HuggingFace cache status & auto-download

Shows which models are already cached. Downloads any that are missing — safe to re-run.


In [3]:
import importlib
importlib.reload(llm_backend)  # picks up any registry changes without restarting the kernel

cache_info = scan_cache_dir()
cached_repos = {r.repo_id: round(r.size_on_disk / 1e9, 2) for r in cache_info.repos}

print("HuggingFace cache (~/.cache/huggingface/hub):")
for repo, gb in sorted(cached_repos.items()):
    print(f"  {repo:<55}  {gb:.2f} GB")

print()
need_download = []
for name in MODELS:
    repo_id = llm_backend._resolve(name)
    status = "✓ cached" if repo_id in cached_repos else "✗ not cached"
    print(f"  {name:<40} → {repo_id}  [{status}]")
    if repo_id not in cached_repos:
        need_download.append((name, repo_id))

if need_download:
    print(f"\nDownloading {len(need_download)} missing model(s)…")
    for name, repo_id in need_download:
        print(f"\n  ↓  {repo_id}  (~4 GB)")
        snapshot_download(repo_id=repo_id, local_files_only=False)
        print(f"  ✓  {name} ready")
else:
    print("\n✓ All models cached — no download needed.")


HuggingFace cache (~/.cache/huggingface/hub):
  Systran/faster-whisper-base                              0.15 GB
  facebook/musicgen-medium                                 8.05 GB
  mlx-community/Mistral-7B-Instruct-v0.3-4bit              4.08 GB
  mlx-community/Qwen2.5-7B-Instruct-4bit                   4.30 GB
  mlx-community/Qwen2.5-VL-7B-Instruct-4bit                5.65 GB
  mlx-community/Qwen3.5-9B-MLX-4bit                        5.98 GB

  mistral:7b                               → mlx-community/Mistral-7B-Instruct-v0.3-4bit  [✓ cached]
  qwen2.5:7b-instruct-q4_K_M               → mlx-community/Qwen2.5-7B-Instruct-4bit  [✓ cached]

✓ All models cached — no download needed.


## Full extraction run

Each model processes all receipts sequentially using the 4-agent pipeline.
Persistence is disabled (`db_path=None`) so nothing is written to disk.

If results come back empty, run the **Step-by-step pipeline debug** section below to pinpoint the failure.


In [4]:
def _fmt_result(result) -> str:
    """Compact single-line summary of extraction result for table display."""
    if not result.success:
        return f"ERROR: {result.error_message}"
    d = result.data
    cp    = d.counterparty.name if d.counterparty else None
    date  = d.receipt_date.date().isoformat() if d.receipt_date else None
    total = str(d.total_amount) if d.total_amount is not None else None
    items = len(d.items) if d.items else 0
    cat   = str(d.category) if d.category else None
    if not cp and not date and not total:
        return "⚠ EMPTY — run Step-by-step debug section below"
    return f"{cp or '—'} | {date or '—'} | {(total or '—') + ' EUR'} | {items} items | {cat or '—'}"


# ── Main extraction loop ──────────────────────────────────────────────────────
raw_results: dict[str, list[dict]] = {}

for model_name, agents_cfg in MODELS.items():
    print(f"\n{'─'*60}")
    print(f"Model: {model_name}  →  {llm_backend._resolve(model_name)}")
    print(f"{'─'*60}")
    agent = FinanceAgent(agents_cfg=agents_cfg, db_path=None)
    rows = []
    for pdf in receipts:
        t0 = time.perf_counter()
        result = agent.process_receipt(pdf)
        elapsed = round(time.perf_counter() - t0, 2)
        summary = _fmt_result(result)
        rows.append({"receipt": pdf.name, "success": result.success, "summary": summary, "time_s": elapsed})
        status = "✓" if result.success else "✗"
        print(f"  {status}  {pdf.name:<35}  {elapsed:>6.2f}s   {summary}")
    raw_results[model_name] = rows

print("\nDone.")



────────────────────────────────────────────────────────────
Model: mistral:7b  →  mlx-community/Mistral-7B-Instruct-v0.3-4bit
────────────────────────────────────────────────────────────
[finamt] adobe-2601.pdf
  [14:37:23] → PyMuPDF ...
  [14:37:23] → PDF text layer (page 1)
  [14:37:23] → Extraction pipeline ...
  [14:37:23] → Agent 1: metadata

[finamt] First run — downloading model mlx-community/Mistral-7B-Instruct-v0.3-4bit
         (~4 GB, cached in ~/.cache/huggingface/hub)



Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[finamt] Model ready.

  [14:37:31] → Agent 2: counterparty
  [14:37:35] → Agent 3: amounts
  [14:37:39] → Agent 4: line items
  ✓  adobe-2601.pdf                        20.97s   Adobe Systems Software Ireland Ltd | 2026-01-30 | 55.84 EUR | 1 items | software
[finamt] anthropic-2601.pdf
  [14:37:44] → PyMuPDF ...
  [14:37:44] → PDF text layer (page 1)
  [14:37:44] → Extraction pipeline ...
  [14:37:44] → Agent 1: metadata
  [14:37:47] → Agent 2: counterparty
  [14:37:50] → Agent 3: amounts
  [14:37:53] → Agent 4: line items
  ✓  anthropic-2601.pdf                    11.54s   Anthropic, PBC | 2026-01-21 | 6.0 EUR | 1 items | other
[finamt] github-2601.pdf
  [14:37:56] → PyMuPDF ...
  [14:37:56] → PDF text layer (page 1)
  [14:37:56] → Extraction pipeline ...
  [14:37:56] → Agent 1: metadata
  [14:37:59] → Agent 2: counterparty
  [14:38:02] → Agent 3: amounts
  [14:38:04] → Agent 4: line items
  ✓  github-2601.pdf                       11.21s   GitHub, Inc. | — | 10.0 EUR | 1 items | sof

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

[finamt] Model ready.

  [14:38:51] → Agent 2: counterparty
  [14:38:55] → Agent 3: amounts
  [14:38:59] → Agent 4: line items
  ✓  adobe-2601.pdf                        20.17s   Adobe Systems Software Ireland Ltd | 2026-01-30 | 55.84 EUR | 1 items | software
[finamt] anthropic-2601.pdf
  [14:39:03] → PyMuPDF ...
  [14:39:03] → PDF text layer (page 1)
  [14:39:03] → Extraction pipeline ...
  [14:39:03] → Agent 1: metadata
  [14:39:06] → Agent 2: counterparty
  [14:39:10] → Agent 3: amounts
  [14:39:13] → Agent 4: line items
  ✓  anthropic-2601.pdf                    12.66s   Anthropic, PBC | 2026-01-21 | 6.0 EUR | 1 items | other
[finamt] github-2601.pdf
  [14:39:15] → PyMuPDF ...
  [14:39:15] → PDF text layer (page 1)
  [14:39:15] → Extraction pipeline ...
  [14:39:15] → Agent 1: metadata
  [14:39:18] → Agent 2: counterparty
  [14:39:22] → Agent 3: amounts
  [14:39:25] → Agent 4: line items
  ✓  github-2601.pdf                       12.95s   GitHub, Inc. | 2026-01-11 | 10.0 EUR | 1 it

## Step-by-step pipeline debug

Run a single receipt through one model and inspect every stage:
OCR text → prompt → raw model output → parsed JSON.
If the full extraction run returns empty results, start here.


In [12]:
# ── Direct backend smoke-test ─────────────────────────────────────────────────
# Tests llm_backend.generate() in isolation with a trivial prompt.
# If this fails, the problem is in the inference layer, not the pipeline.

import importlib
importlib.reload(llm_backend)  # ensure latest code is loaded

TEST_MODEL = "mistral:7b"
repo_id = llm_backend._resolve(TEST_MODEL)
print(f"Testing backend directly")
print(f"  Model  : {TEST_MODEL}")
print(f"  Repo   : {repo_id}")
print(f"  IS_APPLE_SILICON: {llm_backend._IS_APPLE_SILICON}")
print()

try:
    out = llm_backend.generate(
        'Reply with exactly this JSON and nothing else: {"ok": true}',
        TEST_MODEL,
        max_tokens=64,
    )
    print(f"Raw output ({len(out)} chars):")
    print(repr(out))
    print()
    if out.strip():
        print("✓ Backend is generating text correctly")
    else:
        print("⚠  Backend returned empty string — see below for diagnosis")
        if llm_backend._IS_APPLE_SILICON:
            print()
            print("  Possible causes on Apple Silicon / mlx-lm:")
            print("  1. Chat template issue — check tokenizer.apply_chat_template")
            print("  2. max_tokens too low")
            print("  3. Model loaded but tokenizer mis-matched")
            print()
            print("  Trying with verbose=True to see mlx_lm internal output:")
            import mlx_lm
            model, tokenizer = llm_backend._get_mlx(repo_id)
            messages = [{"role": "user", "content": 'Reply with exactly: {"ok": true}'}]
            try:
                formatted = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            except Exception:
                formatted = 'Reply with exactly: {"ok": true}'
            print(f"  Formatted prompt (first 300 chars): {formatted[:300]!r}")
            raw2 = mlx_lm.generate(model, tokenizer, prompt=formatted,
                                   max_tokens=64, verbose=True)
            print(f"  mlx_lm output: {raw2!r}")
except Exception as e:
    print(f"✗ Backend raised an exception: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()


Testing backend directly
  Model  : mistral:7b
  Repo   : mlx-community/Mistral-7B-Instruct-v0.3-4bit
  IS_APPLE_SILICON: True


[finamt] First run — downloading model mlx-community/Mistral-7B-Instruct-v0.3-4bit
         (~4 GB, cached in ~/.cache/huggingface/hub)



Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[finamt] Model ready.

Raw output (12 chars):
'{"ok": true}'

✓ Backend is generating text correctly


In [13]:
import json
import tempfile

# ── Pick a receipt and model to debug ────────────────────────────────────────
DEBUG_RECEIPT = receipts[0]        # change index to try another receipt
DEBUG_MODEL   = "mistral:7b"       # change to any key in MODELS

print(f"Receipt : {DEBUG_RECEIPT.name}")
print(f"Model   : {DEBUG_MODEL}  →  {llm_backend._resolve(DEBUG_MODEL)}")
print()

# ── Step 1: OCR ───────────────────────────────────────────────────────────────
from finamt.ocr_processor import OCRProcessor
from finamt.agents.config import Config

ocr_text = OCRProcessor(Config()).extract_text_from_pdf(DEBUG_RECEIPT)
print("─" * 60)
print("OCR TEXT")
print("─" * 60)
print(ocr_text[:2000] if ocr_text else "⚠  No text extracted — OCR failed")
print()

# ── Step 2: Agent 1 (metadata) call ──────────────────────────────────────────
from finamt.agents.llm_caller import call_llm

# Try both possible module paths for the prompt builder
try:
    from finamt.agents.pipeline import _build_agent1_prompt
except (ImportError, AttributeError):
    from finamt.agents.prompts import build_agent1_prompt as _build_agent1_prompt

cfg = MODELS[DEBUG_MODEL].get_agent_config()
prompt = _build_agent1_prompt(ocr_text)

print("─" * 60)
print("AGENT 1 PROMPT (first 800 chars)")
print("─" * 60)
print(prompt[:800])
print()

with tempfile.TemporaryDirectory() as tmp:
    debug_path = Path(tmp)
    result = call_llm(
        prompt, cfg, "agent1",
        ["receipt_number", "receipt_date", "category"],
        debug_dir=debug_path,
    )
    raw = (debug_path / "agent1_raw.txt").read_text(encoding="utf-8")

print("─" * 60)
print("RAW MODEL OUTPUT")
print("─" * 60)
print(raw[:3000] if raw else "⚠  Empty — generation returned nothing")
print()

print("─" * 60)
print("PARSED RESULT")
print("─" * 60)
print(json.dumps(result, indent=2, ensure_ascii=False) if result else "⚠  None — JSON parsing failed (see raw output above)")


Receipt : adobe-2601.pdf
Model   : mistral:7b  →  mlx-community/Mistral-7B-Instruct-v0.3-4bit

  [14:36:37] → PDF text layer (page 1)


────────────────────────────────────────────────────────────
OCR TEXT
────────────────────────────────────────────────────────────
Rechnungsanschrift
John Doe
10117
GERMANY
USt-IdNr. des Kunden: DE123456789
Rechnung
Positionen
Laufzeit: 30-JAN-2026 bis 26-FEB-2026
Rechnungsinformationen
IEE2026001111111
Rechnungsnummer
Rechnungsdatum
Zahlungsfrist
Bestellnummer
Kundennummer
Kundenauftragsnumm
30-JAN-2026 
Kreditkarte 
ADB011110011DE 
1011111111 
111110111
EUR
Währung
Adobe Systems Software Ireland Ltd
4-6 Riverwalk
Citywest Business Campus
Dublin 24
Ireland
USt-IdNr.: IE6364992H
Seite 1 von 1
Vielen Dank für Ihre Bestellung!
Ansprechpartner
https://helpx.adobe.com/contact.html
ORIGINAL
PRODUKTNUMMER
PRODUKTBESCHREIBUNG
MÉNGE
EINHEIT
EINZELPREIS
GESAMTRABATTBET
RAG/EINHEIT
SUMME NETTO
UST-SATZ
UST
SUMME BRUTTO
65183131
Creative Cloud Pro
1
EA
65.54
(9.70)
55.84
0.00%
0.00
55.84
GESAMT
SUMME NETTO (EUR)
55.84
UST (STEUERSATZ SIEHE OBEN)
0.00
USt
GESAMTBETRAG (EUR)
55.84
Anmerkungen:
Soll

## Results — Side-by-Side Comparison

In [5]:
model_names = list(MODELS.keys())

cols = {}
for m in model_names:
    df = pd.DataFrame(raw_results[m]).set_index("receipt")
    cols[f"{m} — extraction"] = df["summary"]
    cols[f"{m} — time (s)"]   = df["time_s"]

comparison = pd.DataFrame(cols)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 400)
comparison


,mistral:7b — extraction,mistral:7b — time (s),qwen2.5:7b-instruct-q4_K_M — extraction,qwen2.5:7b-instruct-q4_K_M — time (s)
receipt,,,,
adobe-2601.pdf,Adobe Systems Software Ireland Ltd | 2026-01-30 | 55.84 EUR | 1 items | software,20.97,Adobe Systems Software Ireland Ltd | 2026-01-30 | 55.84 EUR | 1 items | software,20.17
anthropic-2601.pdf,"Anthropic, PBC | 2026-01-21 | 6.0 EUR | 1 items | other",11.54,"Anthropic, PBC | 2026-01-21 | 6.0 EUR | 1 items | other",12.66
github-2601.pdf,"GitHub, Inc. | — | 10.0 EUR | 1 items | software",11.21,"GitHub, Inc. | 2026-01-11 | 10.0 EUR | 1 items | services",12.95
microsoft-2601.pdf,Microsoft Ireland Operations Ltd | 2026-01-02 | 13.92 EUR | 1 items | services,24.34,Microsoft Ireland Operations Ltd | 2026-01-02 | 13.92 EUR | 0 items | services,20.26
openai-2601.pdf,"OpenAI, LLC | 2026-01-27 | 10.0 EUR | 1 items | software",11.19,"OpenAI, LLC | 2026-01-27 | 10.0 EUR | 1 items | services",12.77


## Timing Summary

In [7]:
model_names = list(MODELS.keys())

timing = pd.DataFrame({
    "model":     model_names,
    "total_s":   [sum(r["time_s"] for r in raw_results[m]) for m in model_names],
    "avg_s":     [sum(r["time_s"] for r in raw_results[m]) / max(len(receipts), 1) for m in model_names],
    "successes": [sum(r["success"] for r in raw_results[m]) for m in model_names],
}).set_index("model")

timing["total_s"] = timing["total_s"].round(2)
timing["avg_s"]   = timing["avg_s"].round(2)

timing


,total_s,avg_s,successes
model,,,
mistral:7b,79.25,15.85,5
qwen2.5:7b-instruct-q4_K_M,78.81,15.76,5
